# Evaluation Metrics - Object Detection
Master Data & IA, INPT 2026

IoU, Precision / Recall, Average Precision (AP) and mean AP (mAP), implemented
from scratch with numpy on a small synthetic dataset (no download needed).
Boxes use the format (x1, y1, x2, y2).

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib

SAVE_FIGS = "__file__" in globals()  # True when run as a script, False in a notebook
if SAVE_FIGS:
    matplotlib.use("Agg")
    FIG_DIR = Path(__file__).parent / "figures"
    FIG_DIR.mkdir(exist_ok=True)
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CLASSES = ["car", "person", "bike"]
rng = np.random.default_rng(42)


def finish(fig, name):
    """Save the figure when run as a script, display it in a notebook."""
    fig.tight_layout()
    if SAVE_FIGS:
        fig.savefig(FIG_DIR / name, dpi=150)
        plt.close(fig)
    else:
        plt.show()

## 1. Intersection over Union (IoU)
IoU = area of overlap / area of union. It decides whether a predicted box matches a ground-truth box.

In [ ]:
def iou(a, b):
    """IoU between two boxes (x1, y1, x2, y2)."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-9)


gt_box = (0, 0, 100, 100)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, dx in zip(axes, (5, 33, 82)):
    pred = (dx, 0, 100 + dx, 100)
    ax.add_patch(Rectangle((0, 0), 100, 100, fill=False, ec="#13A89E", lw=2.5, label="ground truth"))
    ax.add_patch(Rectangle((dx, 0), 100, 100, fill=False, ec="#E8751A", lw=2.5, ls="--", label="prediction"))
    ax.set(xlim=(-10, 200), ylim=(-10, 110), aspect="equal", xticks=[], yticks=[])
    ax.set_title(f"IoU = {iou(gt_box, pred):.2f}")
axes[0].legend(loc="lower left", fontsize=8)
finish(fig, "iou_examples.png")

## 2. Synthetic dataset
60 images, 3 classes. The detector finds ~85 % of objects with noisy boxes,
sometimes predicts the wrong class, and adds some false positives.

In [ ]:
def make_dataset(n_images=60):
    gts, dets = {}, []
    for img in range(n_images):
        gts[img] = []
        for _ in range(rng.integers(1, 5)):
            x1, y1 = rng.uniform(0, 400, 2)
            w, h = rng.uniform(40, 160, 2)
            gts[img].append((int(rng.integers(0, 3)), (x1, y1, x1 + w, y1 + h)))
        for cls, box in gts[img]:
            if rng.random() < 0.85:  # object detected
                jitter = rng.normal(0, 8, 4)
                score = float(np.clip(1 - np.abs(jitter).mean() / 40 + rng.normal(0, 0.1), 0.05, 0.99))
                pred_cls = cls if rng.random() < 0.92 else int(rng.integers(0, 3))
                dets.append((img, pred_cls, score, tuple(np.array(box) + jitter)))
        for _ in range(rng.integers(0, 3)):  # false positives
            x1, y1 = rng.uniform(0, 400, 2)
            w, h = rng.uniform(40, 160, 2)
            dets.append((img, int(rng.integers(0, 3)), float(rng.uniform(0.05, 0.7)), (x1, y1, x1 + w, y1 + h)))
    return gts, dets


gts, dets = make_dataset()
print(f"{sum(len(v) for v in gts.values())} ground-truth boxes, {len(dets)} detections")

## 3. Matching, Precision / Recall and Average Precision
Detections of one class are sorted by confidence. Each one is a **TP** if it overlaps
an unmatched ground-truth box with IoU >= threshold, otherwise a **FP**.
Unmatched ground-truth boxes are **FN**. AP is the area under the Precision-Recall curve.

In [ ]:
def evaluate_class(cls, iou_thr=0.5):
    gt = {i: [b for c, b in lst if c == cls] for i, lst in gts.items()}
    n_pos = sum(len(v) for v in gt.values())
    d = sorted([x for x in dets if x[1] == cls], key=lambda x: -x[2])
    used = {i: np.zeros(len(v), bool) for i, v in gt.items()}
    tp = np.zeros(len(d))
    for k, (img, _, _, box) in enumerate(d):
        if gt[img]:
            ious = [iou(box, g) for g in gt[img]]
            j = int(np.argmax(ious))
            if ious[j] >= iou_thr and not used[img][j]:
                tp[k], used[img][j] = 1, True
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    recall = ctp / max(n_pos, 1)
    precision = ctp / np.maximum(ctp + cfp, 1e-9)
    # all-point interpolated AP (area under the monotone precision envelope)
    mrec = np.concatenate([[0], recall, [1]])
    mpre = np.concatenate([[0], precision, [0]])
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
    return dict(ap=ap, precision=precision, recall=recall, tp=tp,
                scores=np.array([x[2] for x in d]), n_pos=n_pos)


print("AP per class at IoU = 0.5")
res50 = {}
for c, name in enumerate(CLASSES):
    res50[c] = evaluate_class(c, 0.5)
    print(f"  {name:>6}: AP = {res50[c]['ap']:.4f}")

## 4. Precision / Recall / F1 at a fixed confidence threshold (0.5, IoU 0.5)

In [ ]:
CONF = 0.5
TP = FP = FN = 0
for c in range(3):
    r = res50[c]
    keep = r["scores"] >= CONF
    tp_c = int(r["tp"][keep].sum())
    TP += tp_c
    FP += int(keep.sum()) - tp_c
    FN += r["n_pos"] - tp_c
P, R = TP / (TP + FP), TP / (TP + FN)
F1 = 2 * P * R / (P + R)
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Precision={P:.4f}  Recall={R:.4f}  F1={F1:.4f}")

## 5. Precision-Recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for c, name in enumerate(CLASSES):
    ax.plot(res50[c]["recall"], res50[c]["precision"], lw=2, label=f"{name} (AP={res50[c]['ap']:.3f})")
ax.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1.02), ylim=(0, 1.05),
       title="Precision-Recall curves (IoU = 0.5)")
ax.legend(loc="lower left")
finish(fig, "pr_curve.png")

## 6. mAP@0.5 and mAP@[0.5:0.95] (COCO style)
mAP = mean of AP over classes. COCO also averages over 10 IoU thresholds (0.50, 0.55, ..., 0.95).

In [ ]:
thresholds = np.round(np.arange(0.5, 0.96, 0.05), 2)
map_per_thr = [np.mean([evaluate_class(c, t)["ap"] for c in range(3)]) for t in thresholds]
for t, m in zip(thresholds, map_per_thr):
    print(f"  mAP@{t:.2f} = {m:.4f}")
print(f"\nmAP@0.5       = {map_per_thr[0]:.4f}")
print(f"mAP@0.75      = {map_per_thr[list(thresholds).index(0.75)]:.4f}")
print(f"mAP@[.5:.95]  = {np.mean(map_per_thr):.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([f"{t:.2f}" for t in thresholds], map_per_thr, color="#13A89E")
ax.set(xlabel="IoU threshold", ylabel="mAP", ylim=(0, 1.05), title="mAP at each IoU threshold")
ax.tick_params(axis="x", labelsize=8)
finish(fig, "map_vs_iou.png")